In [1]:
# Project set up

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

g:\My Drive\Graduação e Pós\USP\MBA USP IA e Big Data\TCC\port_leadtime_prediction


In [2]:
# Imports

import pandas as pd

from src.data_sources.port_call import load_port_call_files, process_port_call
from src.processing.master_table import build_master_calls
from src.processing.validation import (
    add_event_presence_flags,
    add_temporal_consistency_flags,
    add_duration_check_columns,
    add_extreme_duration_flags,
    define_eda_eligibility,
    build_quality_summary,
)

In [3]:
# Load and process data

RAW_PORT_CALL_DIR = PROJECT_ROOT / "data" / "raw" / "estadia"

df_raw = load_port_call_files(RAW_PORT_CALL_DIR)
df_port_call = process_port_call(df_raw)

print(df_port_call.shape)
df_port_call.head()

(162036, 23)


,port_call_id,port,port_name,imo,vessel_id,vessel_name,boarding_crew,transit_crew,unboarding_crew,total_crew,...,total_passengers,operation_type,arrival_port_ts,berthing_ts,unberthing_ts,departure_port_ts,source_port,source_port_name,destination_port,destination_port_name
0,12022,BRSSZ,SANTOS,9634854,<NA>,ORION GLOBE,0.0,21,0,21,...,0,Carga,2022-01-13 06:00:00,2022-01-20 17:20:00,2022-01-23 21:52:00,2022-01-23 21:52:00,SGSIN,SINGAPURA (SINGAPORE),SGSIN,SINGAPURA (SINGAPORE)
1,22022,BRSSZ,SANTOS,9699359,<NA>,COPENHAGEN EAGLE,0.0,19,0,19,...,0,Descarga,2022-01-15 09:05:00,2022-02-25 05:05:00,2022-02-28 13:45:00,2022-02-28 13:45:00,SGSIN,SINGAPURA (SINGAPORE),BRPNG,PARANAGUA
2,32022,BRSSZ,SANTOS,9686273,<NA>,CEPHEUS OCEAN,0.0,19,0,19,...,0,Descarga,2022-01-19 10:00:00,2022-02-28 20:25:00,NaT,NaT,RU006,UST - LUGA,<NA>,<NA>
3,62022,BRRJ022,PORTO DO AÇU,9644158,<NA>,ZARAPITO,0.0,15,0,15,...,0,Off-shore,2022-01-01 08:00:00,2022-01-01 08:00:00,2022-01-03 10:27:00,2022-01-03 10:27:00,BRRIO,RIO DE JANEIRO,BRRIO,RIO DE JANEIRO
4,72022,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ,9618757,3813885801,MAR LIMPO II,0.0,13,0,13,...,0,"Desembarque/Embarque de Passageiros,Off-shore",2022-01-01 17:45:00,2022-01-03 06:50:00,2022-01-04 03:00:00,2022-01-04 17:40:00,BRRIO,RIO DE JANEIRO,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ


In [4]:
# Build master table

df_master = build_master_calls(df_port_call)

print(df_master.shape)
df_master.head()

(146037, 18)


,port_call_id,port,port_name,imo,vessel_id,vessel_name,operation_type,source_port,source_port_name,destination_port,destination_port_name,arrival_port_ts,berthing_ts,unberthing_ts,departure_port_ts,port_display,source_port_display,destination_port_display
0,561202021,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,JURUTI,2022-01-02 00:24:00,2022-01-02 00:24:00,2022-01-03 02:05:00,2022-01-03 02:05:00,BR052001 - TERMINAL FLUVIAL DE JURUTI,BRAMW001 - TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052 - JURUTI
1,4172022,BR052001,TERMINAL FLUVIAL DE JURUTI,9304241,<NA>,GOOD HOPE MAX,Carga,BR052,JURUTI,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,2022-01-06 23:54:00,2022-01-06 23:54:00,2022-01-08 17:54:00,2022-01-08 17:54:00,BR052001 - TERMINAL FLUVIAL DE JURUTI,BR052 - JURUTI,BRAMW001 - TERMINAL DA ALUMAR - SÃO LUIZ - MA
2,7012022,BR052001,TERMINAL FLUVIAL DE JURUTI,9628922,<NA>,AMBERJACK,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,2022-01-08 21:06:00,2022-01-08 21:06:00,2022-01-10 01:42:00,2022-01-10 01:42:00,BR052001 - TERMINAL FLUVIAL DE JURUTI,BRAMW001 - TERMINAL DA ALUMAR - SÃO LUIZ - MA,BRAMW001 - TERMINAL DA ALUMAR - SÃO LUIZ - MA
3,12292022,BR052001,TERMINAL FLUVIAL DE JURUTI,9473339,<NA>,JURUTI,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,JURUTI,2022-01-13 11:50:00,2022-01-13 11:50:00,2022-01-14 21:07:00,2022-01-14 21:07:00,BR052001 - TERMINAL FLUVIAL DE JURUTI,BRAMW001 - TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052 - JURUTI
4,19472022,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BR052,JURUTI,BR052,JURUTI,2022-01-17 00:18:00,2022-01-17 00:18:00,2022-01-18 01:06:00,2022-01-18 01:06:00,BR052001 - TERMINAL FLUVIAL DE JURUTI,BR052 - JURUTI,BR052 - JURUTI


In [5]:
# Apply validation rules

df_qc = add_event_presence_flags(df_master)
df_qc = add_temporal_consistency_flags(df_qc)
df_qc = add_duration_check_columns(df_qc)
df_qc = add_extreme_duration_flags(df_qc)
df_qc = define_eda_eligibility(df_qc)

print(df_qc.shape)
df_qc.head()

(146037, 37)


,port_call_id,port,port_name,imo,vessel_id,vessel_name,operation_type,source_port,source_port_name,destination_port,...,tmp_post_operation_h,tmp_total_port_stay_h,flag_negative_tmp_wait_for_berthing_h,flag_negative_tmp_operation_h,flag_negative_tmp_post_operation_h,flag_negative_tmp_total_port_stay_h,flag_wait_too_long,flag_operation_too_long,flag_total_too_long,eligible_for_eda
0,561202021,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,0.0,25.683333,False,False,False,False,False,False,False,True
1,4172022,BR052001,TERMINAL FLUVIAL DE JURUTI,9304241,<NA>,GOOD HOPE MAX,Carga,BR052,JURUTI,BRAMW001,...,0.0,42.000000,False,False,False,False,False,False,False,True
2,7012022,BR052001,TERMINAL FLUVIAL DE JURUTI,9628922,<NA>,AMBERJACK,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BRAMW001,...,0.0,28.600000,False,False,False,False,False,False,False,True
3,12292022,BR052001,TERMINAL FLUVIAL DE JURUTI,9473339,<NA>,JURUTI,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,0.0,33.283333,False,False,False,False,False,False,False,True
4,19472022,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BR052,JURUTI,BR052,...,0.0,24.800000,False,False,False,False,False,False,False,True


In [6]:
# QA Summary

quality_summary = build_quality_summary(df_qc)
quality_summary

,metric,value
0,total_rows,146037
1,unique_port_call_id,146037
2,missing_arrival_port_ts,8
3,missing_berthing_ts,3
4,missing_unberthing_ts,3892
5,missing_departure_port_ts,4105
6,arrival_after_berthing,1
7,berthing_after_unberthing,0
8,unberthing_after_departure,0
9,negative_wait_for_berthing,1


In [7]:
# Checking eligibility criteria for EDA base

df_qc["eligible_for_eda"].value_counts(dropna=False)

eligible_for_eda
True     141895
False      4142
Name: count, dtype: int64

In [8]:
# Checking temporal inconsistencies

cols_to_show = [
    "port_call_id",
    "port_name",
    "vessel_name",
    "arrival_port_ts",
    "berthing_ts",
    "unberthing_ts",
    "departure_port_ts",
    "flag_arrival_after_berthing",
    "flag_berthing_after_unberthing",
    "flag_unberthing_after_departure",
]

df_qc.loc[
    df_qc[
        [
            "flag_arrival_after_berthing",
            "flag_berthing_after_unberthing",
            "flag_unberthing_after_departure",
        ]
    ].fillna(False).any(axis=1),
    cols_to_show,
].head(20)

,port_call_id,port_name,vessel_name,arrival_port_ts,berthing_ts,unberthing_ts,departure_port_ts,flag_arrival_after_berthing,flag_berthing_after_unberthing,flag_unberthing_after_departure
125732,188242025,SANTOS,TYMFI,2025-05-02 19:13:00,2025-04-30 14:10:00,NaT,NaT,True,False,False


In [9]:
# Checking negative durations

duration_cols = [
    "port_call_id",
    "port_name",
    "vessel_name",
    "tmp_wait_for_berthing_h",
    "tmp_operation_h",
    "tmp_post_operation_h",
    "tmp_total_port_stay_h",
    "flag_negative_tmp_wait_for_berthing_h",
    "flag_negative_tmp_operation_h",
    "flag_negative_tmp_post_operation_h",
    "flag_negative_tmp_total_port_stay_h",
]

df_qc.loc[
    df_qc[
        [
            "flag_negative_tmp_wait_for_berthing_h",
            "flag_negative_tmp_operation_h",
            "flag_negative_tmp_post_operation_h",
            "flag_negative_tmp_total_port_stay_h",
        ]
    ].fillna(False).any(axis=1),
    duration_cols,
].head(20)

,port_call_id,port_name,vessel_name,tmp_wait_for_berthing_h,tmp_operation_h,tmp_post_operation_h,tmp_total_port_stay_h,flag_negative_tmp_wait_for_berthing_h,flag_negative_tmp_operation_h,flag_negative_tmp_post_operation_h,flag_negative_tmp_total_port_stay_h
125732,188242025,SANTOS,TYMFI,-53.05,NaN,NaN,NaN,True,False,False,False


In [10]:
# Checking missing main timestamps

main_ts_cols = [
    "arrival_port_ts",
    "berthing_ts",
    "unberthing_ts",
    "departure_port_ts",
]

df_qc[main_ts_cols].isna().mean().sort_values(ascending=False)

departure_port_ts    0.028109
unberthing_ts        0.026651
arrival_port_ts      0.000055
berthing_ts          0.000021
dtype: float64

In [11]:
interim_dir = PROJECT_ROOT / "data" / "interim"
tables_dir = PROJECT_ROOT / "outputs" / "tables"

interim_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

qc_path = interim_dir / "master_port_calls_qc.parquet"
summary_path = tables_dir / "port_call_quality_summary.csv"

df_qc.to_parquet(qc_path, index=False)
quality_summary.to_csv(summary_path, index=False)

print(f"QC base saved to: {qc_path}")
print(f"Quality summary saved to: {summary_path}")

QC base saved to: g:\My Drive\Graduação e Pós\USP\MBA USP IA e Big Data\TCC\port_leadtime_prediction\data\interim\master_port_calls_qc.parquet
Quality summary saved to: g:\My Drive\Graduação e Pós\USP\MBA USP IA e Big Data\TCC\port_leadtime_prediction\outputs\tables\port_call_quality_summary.csv
